# 04 – Profiles

Esplorazione e data cleaning del dataset `profiles.csv`.

**Colonne:**
| Colonna | Descrizione |
|---|---|
| `username` | Nome utente MAL (chiave primaria) |
| `gender` | Genere dichiarato |
| `birthday` | Data di nascita |
| `location` | Paese dell'utente |
| `joined` | Data di iscrizione a MAL |
| `watching` | Numero di anime attualmente in visione |
| `completed` | Numero di anime completati |
| `on_hold` | Numero di anime in pausa |
| `dropped` | Numero di anime abbandonati |
| `plan_to_watch` | Numero di anime in lista d'attesa |

## 1. Import e caricamento dati

Importiamo le librerie necessarie e carichiamo il file csv. Facciamo una esplorazione generica per capire la struttura e le caratteristiche del dataset.

In [ ]:
import pandas as pd
import numpy as np
from dataset_analyzer import analyze

df_pr = pd.read_csv('../datasets/profiles.csv')
print(f'Shape: {df_pr.shape}')
print()
df_pr.info()
df_pr.head()

Il dataset contiene **337.155 righe** e **10 colonne**. I tipi di dati richiedono conversione: le colonne statistiche come `watching`, `completed`, etc sono `str` invece di `int64`, e `birthday` e `joined` sono stringhe invece di `datetime`.

## 1.1 Rimozione duplicati esatti

Prima dell'analisi per colonna, rimuoviamo le righe con valori identici in **tutte** le colonne, mantenendo solo la prima occorrenza.

In [ ]:
n_originale = len(df_pr)

mask_dup = df_pr.duplicated(keep=False)
n_righe_coinvolte = mask_dup.sum()
n_gruppi = df_pr[mask_dup].duplicated(keep='first').sum()
n_tenute = n_righe_coinvolte - n_gruppi

print(f'Righe totali coinvolte in duplicazioni : {n_righe_coinvolte:,}')
print(f'  → prime occorrenze mantenute         : {n_tenute:,}')
print(f'  → occorrenze extra rimosse           : {n_gruppi:,}')
print()

df_pr.drop_duplicates(keep='first', inplace=True)
print(f'Righe prima della rimozione : {n_originale:,}')
print(f'Righe dopo la rimozione     : {len(df_pr):,}')

## 1.2 Rimozione profili vuoti

Verifichiamo la presenza di profili utente senza nessun dato statistico.

In [ ]:
stat_cols = ['watching', 'completed', 'on_hold', 'dropped', 'plan_to_watch']
mask_no_stats = df_pr[stat_cols].isna().all(axis=1)
print(f'Profili senza dati statistici : {mask_no_stats.sum():,}')
print(f'  di cui anche senza joined   : {(mask_no_stats & df_pr["joined"].isna()).sum():,}')
print()
with pd.option_context('display.expand_frame_repr', False, 'display.max_columns', None):
    print('Esempio profili vuoti:')
    print(df_pr[mask_no_stats][['username', 'gender', 'location', 'joined'] + stat_cols].head())
    print()
    print('Esempio profili vuoti con joined:')
    print(df_pr[mask_no_stats & df_pr['joined'].notna()][['username', 'gender', 'location', 'joined'] + stat_cols].head())

Dalla verifica risultano 1,678 profili senza dati statistici da cui 1,676 senza la data `joined`. Abbiamo deciso di rimuoverli in quanto non contribuiscono a nessun dato statistico e solo la location in se è un dato poco informativo.

In [ ]:
n_prima = len(df_pr)
df_pr = df_pr[~mask_no_stats].copy()
print(f'Righe prima della rimozione : {n_prima:,}')
print(f'Righe dopo la rimozione     : {len(df_pr):,}')